# 第 8 章 · 记忆（长期：MemoryService）

`MemoryService` 是**长期记忆**的接口，只有两个方向：

- 写：`add_session_to_memory(session)` —— 把一次会话抽取进长期库
- 读：`search_memory(app_name, user_id, query)` —— 语义检索

书里只给了构造代码。下面**真做一次**：跑一个 agent → 把会话存进记忆 → 再检索回来。
所以你能看到"记忆"到底长什么样。

示例对话用英文：`InMemoryMemoryService` 按空格切词，中文没有空格，整句会被当成一个词（见 `../../my_notes/ch08_memory.md` 的小记）。

kernel 选 `Python (agentic-adk)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

import uuid
from google.genai import types
from google.adk.runners import InMemoryRunner

import asyncio
import threading

from google.adk.agents import LlmAgent
from google.adk.memory import InMemoryMemoryService

# ---- 书里的两种实现 ----
# ① 内存版：进程内有效，重启即失（适合测试）
memory_service = InMemoryMemoryService()

# 【适配】② Vertex AI RAG 版需要 GCP 项目 + 认证 + RAG Corpus，本机不可用 → 只做说明
try:
    from google.adk.memory import VertexAiRagMemoryService
    print("② VertexAiRagMemoryService 可导入，但需要 GCP 的 RAG Corpus，本机不调用。")
except Exception as e:      # noqa: BLE001
    print("② VertexAiRagMemoryService 不可用:", type(e).__name__)




# ---- 真做一次：跑一轮对话 → 写入长期记忆 → 检索 ----
USER = "u1"

talk_agent = LlmAgent(
    name="Talker",
    model=adk_model(),
    instruction="Answer in one short sentence.",
)

runner = InMemoryRunner(talk_agent)
APP = runner.app_name                     # 必须用 runner 自己的 app_name，否则读不到 session
sid = str(uuid.uuid4())
runner.session_service.create_session_sync(app_name=APP, user_id=USER, session_id=sid)
for _ in retry_call(lambda: list(runner.run(
        user_id=USER, session_id=sid,
        new_message=types.Content(role="user", parts=[types.Part(text="I am flying to New York next week and I prefer a window seat.")]))),
        label="run "):
    pass
session = runner.session_service.get_session_sync(app_name=APP, user_id=USER, session_id=sid)

run_async(memory_service.add_session_to_memory(session))
print("\n已把这次会话写入长期记忆；session 里有", len(session.events), "个事件")

# ★ 检索是长期记忆真正的难点：InMemoryMemoryService 用的是词匹配（不是语义），
#   命中数取决于查询与原文共享多少个词 —— 换个问法，结果就不一样。
for query in ("window seat", "Which seat does the user prefer?"):
    resp = run_async(memory_service.search_memory(app_name=APP, user_id=USER, query=query))
    hits = getattr(resp, "memories", []) or []
    print(f"\nquery={query!r} -> 命中 {len(hits)} 条")
    for m in hits[:1]:
        parts = getattr(getattr(m, "content", None), "parts", None) or []
        text = "".join(p.text for p in parts if getattr(p, "text", None))
        print("  -", text.strip()[:160])

print("\n（记忆的成败在检索，不在存储）")


② VertexAiRagMemoryService 可导入，但需要 GCP 的 RAG Corpus，本机不调用。



已把这次会话写入长期记忆；session 里有 2 个事件

query='window seat' -> 命中 2 条
  - I am flying to New York next week and I prefer a window seat.

query='Which seat does the user prefer?' -> 命中 2 条
  - I am flying to New York next week and I prefer a window seat.

（记忆的成败在检索，不在存储）
